In [1]:
! git clone https://github.com/zakariaaithssain/SegKD.git
%cd SegKD

Cloning into 'SegKD'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 76 (delta 9), reused 20 (delta 5), pack-reused 51 (from 1)
Receiving objects: 100% (76/76), 117.26 MiB | 38.97 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/kaggle/working/SegKD


In [2]:
!pip install -q albumentations tabulate segmentation-models-pytorch kagglehub google

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.0 MB/s eta 0:00:00


In [3]:
import kagglehub
path = kagglehub.dataset_download("lakshaymiddha/crack-segmentation-dataset") + "/crack_segmentation_dataset"
print("Path to dataset files:", path)
import os, shutil, random

src_images = f'{path}/images'
src_masks  = f'{path}/masks'

random.seed(42)
files = sorted(os.listdir(src_images))
random.shuffle(files)

n       = len(files)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

splits = {
    'train' : files[:n_train],
    'val'   : files[n_train:n_train + n_val],
    'test'  : files[n_train + n_val:],
}

for split, subset in splits.items():
    os.makedirs(f'data/{split}/images', exist_ok=True)
    os.makedirs(f'data/{split}/masks',  exist_ok=True)
    for f in subset:
        mask_f = f if os.path.exists(f'{src_masks}/{f}') else f.replace('.jpg', '.png')
        shutil.copy(f'{src_images}/{f}',     f'data/{split}/images/{f}')
        shutil.copy(f'{src_masks}/{mask_f}', f'data/{split}/masks/{mask_f}')
    print(f'{split}: {len(subset)} images')
     

Path to dataset files: /kaggle/input/datasets/lakshaymiddha/crack-segmentation-dataset/crack_segmentation_dataset
train: 7908 images
val: 1694 images
test: 1696 images


In [4]:
!python src/train.py --mode distill --epochs 10


  Device : cuda
[train] 7908 images chargées
[val] 1694 images chargées
[test] 1696 images chargées

  MODE : DISTILLATION (Feature-Based KD)
config.json: 100%|██████████████████████████████| 156/156 [00:00<00:00, 770kB/s]
model.safetensors: 100%|███████████████████| 87.3M/87.3M [00:01<00:00, 77.7MB/s]
  ✓ Checkpoint chargé ← checkpoints/teacher_best.pth  (epoch 17, IoU 0.6090)
config.json: 100%|██████████████████████████████| 106/106 [00:00<00:00, 595kB/s]
model.safetensors: 100%|███████████████████| 14.2M/14.2M [00:00<00:00, 34.0MB/s]
  Epoch [001/10]  Loss: 0.9799  (seg=0.4530, kd=0.5269)  IoU: 0.5108  F1: 0.6478
  ✓ Checkpoint sauvegardé → checkpoints/student_distilled_best.pth
  Epoch [002/10]  Loss: 0.4782  (seg=0.2381, kd=0.2402)  IoU: 0.5461  F1: 0.6800
  ✓ Checkpoint sauvegardé → checkpoints/student_distilled_best.pth
  Epoch [003/10]  Loss: 0.4040  (seg=0.2099, kd=0.1941)  IoU: 0.5439  F1: 0.6789
  Epoch [004/10]  Loss: 0.3742  (seg=0.2005, kd=0.1737)  IoU: 0.5574  F1: 0.691